# Lab 21 — Nguyễn Nhân Sâm · 2A202602672

Chạy từng ô theo thứ tự. Chọn **Runtime → Change runtime type → T4 GPU** trước ô 1.
Notebook lấy code từ **fork cá nhân**, nhánh `feature/lab21-finetuning`.
Bản chính thức dùng **EPOCHS=2**, **full eval 50 target + 15 regression**; không bật EVAL_LIMIT.

1. Setup và smoke.
2. NB1: mask/template/p95.
3. NB2: baseline trước train, lưu prediction và manifest.
4. NB3: correct.
5. NB4: ba đối chứng.
6. NB5: score, verdict và paired qualitative.
7. Xuất checkpoint để đưa về máy, rồi hoàn thiện report và chạy verify cuối.

**Model FAILED vẫn là kết quả hợp lệ.** Không sửa eval, baseline hoặc gate để đổi kết quả.
Colab /content là tạm thời: hãy tải checkpoint sau các chặng quan trọng.

In [ ]:
# 1. Setup — fork cá nhân và cấu hình chính thức
import os, pathlib, subprocess, sys, json
REPO = 'https://github.com/Nguyen-Sam-sheep-zzz/Day21-Track3-Finetuning-Lab.git'
BRANCH = 'feature/lab21-finetuning'
ROOT = pathlib.Path('/content/Lab21-Sam')
if not ROOT.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO, str(ROOT)], check=True)
if not (ROOT / '.git').is_dir():
    raise RuntimeError('Thư mục /content/Lab21-Sam đã tồn tại nhưng không phải checkout bài lab.')
os.chdir(ROOT)
actual_remote = subprocess.check_output(['git', 'remote', 'get-url', 'origin'], text=True).strip()
actual_branch = subprocess.check_output(['git', 'branch', '--show-current'], text=True).strip()
if actual_remote != REPO or actual_branch != BRANCH:
    raise RuntimeError('Checkout khác fork/nhánh được chọn. Giữ nguyên dữ liệu cũ, dùng runtime mới.')
# Không pull giữa một thí nghiệm đã đóng băng.
if not (ROOT / 'results/baselines_frozen.json').exists() and not (ROOT / 'adapters/correct/adapter_model.safetensors').exists():
    subprocess.run(['git', 'pull', '--ff-only'], check=True)
os.environ['COMPUTE_TIER'] = 'T4'
os.environ['MASK_MODE'] = 'assistant-only'
os.environ['EPOCHS'] = '2'
os.environ.pop('EVAL_LIMIT', None)
os.environ.pop('BASE_MODEL', None)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
(ROOT / 'results').mkdir(exist_ok=True)
with (ROOT / 'results/environment_t4.txt').open('w', encoding='utf-8') as fh:
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=fh, check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Chưa có GPU. Chọn Runtime → Change runtime type → T4 GPU.')
if 'T4' not in torch.cuda.get_device_name(0):
    raise RuntimeError('Notebook chính thức này chọn T4. Kiểm tra loại GPU trước khi bắt đầu.')
print('GPU:', torch.cuda.get_device_name(0))
print('commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
print('epochs=2; eval=full; model=unsloth/Qwen3.5-4B')

In [ ]:
# Hàm chạy có log và xuất checkpoint (chỉ file bài lab, không chứa .env/cache/full merged model)
import datetime, zipfile

def export_checkpoint(label):
    stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    archive = pathlib.Path('/content') / f'lab21_2A202602672_{label}_{stamp}.zip'
    files = []
    for directory in ['results', 'notebooks', 'src/labkit', 'scripts', 'colab', 'submission']:
        for p in (ROOT / directory).rglob('*'):
            if p.is_file() and '__pycache__' not in p.parts and p.suffix in {'.json', '.csv', '.txt', '.md', '.py', '.ipynb'}:
                files.append(p)
    for p in (ROOT / 'adapters').glob('*/adapter*'):
        if p.is_file() and p.suffix in {'.json', '.safetensors'}:
            files.append(p)
    for name in ['requirements.txt', 'requirements-cpu.txt', 'LAB21_PLAN.md', 'PROGRESS_LAB.md', 'HUONG_DAN_COLAB.md']:
        p = ROOT / name
        if p.is_file():
            files.append(p)
    with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as zf:
        for p in sorted(set(files)):
            zf.write(p, 'lab21_2A202602672/' + p.relative_to(ROOT).as_posix())
    print('Checkpoint:', archive, '| MB:', round(archive.stat().st_size / 1024**2, 1))
    return archive

def run_stage(stage, script=None):
    started = datetime.datetime.now(datetime.timezone.utc).isoformat()
    logfile = ROOT / 'results' / f'colab_{stage}.txt'
    argv = [sys.executable, '-u', 'scripts/colab_run.py', stage]
    if script:
        argv = [sys.executable, '-u', *script]
    with logfile.open('a', encoding='utf-8') as fh:
        fh.write(f'\nSTART {started} stage={stage}\n')
        with subprocess.Popen(argv, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, encoding='utf-8', errors='replace',
                              env={**os.environ, 'PYTHONUNBUFFERED': '1', 'PYTHONUTF8': '1'}) as proc:
            for line in proc.stdout:
                print(line, end='')
                fh.write(line)
                fh.flush()
            returncode = proc.wait()
        ended = datetime.datetime.now(datetime.timezone.utc).isoformat()
        fh.write(f'END {ended} stage={stage} exit={returncode}\n')
    archive = export_checkpoint(stage if returncode == 0 else stage + '_failed')
    if returncode:
        raise RuntimeError(f'{stage} thất bại, exit={returncode}. Giữ checkpoint để chẩn đoán; chưa chạy bước tiếp.')
    return archive

run_stage('smoke', ['scripts/verify.py', '--smoke'])

## 2. NB1 — Mask, template và độ dài
Đọc đoạn supervised: answer phải có, question phải không có. Giữ max_length=1024 của tier và ghi lý do nếu p95 gợi ý khác.

In [ ]:
run_stage('nb1')
proof = json.loads((ROOT / 'results/mask_proof.json').read_text(encoding='utf-8'))
assert proof['answer_is_supervised'] and proof['question_is_masked']
assert proof['supervised_fraction'] < 0.95
print(json.loads((ROOT / 'results/token_stats.json').read_text(encoding='utf-8')))

## 3. NB2 — Đóng băng baseline trước training
Chạy lần đầu trước NB3. Không chạy lại NB2 sau khi có adapter. Nếu có lỗi môi trường, lưu checkpoint và gửi log để xử lý.

In [ ]:
run_stage('nb2')
frozen = json.loads((ROOT / 'results/baselines_frozen.json').read_text(encoding='utf-8'))
assert not frozen.get('smoke_mode') and frozen['n_target'] == 50 and frozen['n_regression'] == 15
print('baseline a:', frozen['baseline_a'])
print('baseline b:', frozen['baseline_b'])
assert frozen['baseline_b']['target'] > frozen['baseline_a']['target'], 'Baseline b chưa mạnh hơn a. Dừng trước training.'

In [ ]:
# Tải checkpoint baseline; giữ bản này để chứng minh thời điểm trước training.
from google.colab import files
files.download(str(export_checkpoint('before_training')))

## 4. NB3 — Adapter correct
Dùng mask đã proof, r=16, alpha=32, LR=1e-4, cùng recipe step với NB4. Không xếp hạng bằng train loss.

In [ ]:
run_stage('nb3')
assert (ROOT / 'adapters/correct/adapter_model.safetensors').is_file()
print((ROOT / 'results/runs.csv').read_text(encoding='utf-8'))

In [ ]:
# Tải checkpoint sau correct để giảm nguy cơ mất training khi runtime bị ngắt.
files.download(str(export_checkpoint('after_correct')))

## 5. NB4 — Ba đối chứng
Nếu phiên cũ đã hoàn thành một adapter, NB4 tự skip. Phải kiểm tra adapter đó thuộc đúng thí nghiệm này; không copy adapter từ smoke/config khác.

In [ ]:
run_stage('nb4')
import csv
with (ROOT / 'results/runs.csv').open(encoding='utf-8') as fh:
    rows = list(csv.DictReader(fh))
latest = {row['run']: row for row in rows}
keys = ['correct', 'attn_only', 'wrong_lr', 'qlora']
assert all(key in latest for key in keys), 'Thiếu run đối chứng.'
assert len({latest[key]['max_steps'] for key in keys}) == 1, 'Ngân sách step không khớp.'
correct_params = int(latest['correct']['trainable_params'])
assert abs(int(latest['attn_only']['trainable_params']) - correct_params) / correct_params < 0.05
print('Bốn run đầy đủ, cùng max_steps, tham số attn_only khớp <5%.')
files.download(str(export_checkpoint('after_contrasts')))

## 6. NB5 — Đánh giá và phán quyết
Đọc comparison và autopsy. Ví dụ định tính phải so baseline b với FT trên cùng ticket; không bịa ca thắng/thua.

In [ ]:
run_stage('nb5')
verdict = json.loads((ROOT / 'results/verdict.json').read_text(encoding='utf-8'))
print(json.dumps(verdict, ensure_ascii=False, indent=2))
autopsy = json.loads((ROOT / 'results/autopsy.json').read_text(encoding='utf-8'))
assert {row['run'] for row in autopsy} >= {'correct', 'attn_only', 'wrong_lr', 'qlora'}
print(json.dumps(autopsy, ensure_ascii=False, indent=2))
print('Kết quả đã có. Report hiện còn phần chưa đo ở bản ban đầu; cần viết lại theo số thực tế.')

## 7. Xuất kết quả và hoàn thiện report
Tải ZIP dưới đây về `C:\Users\SAM\IdeaProjects\VinUniAIThucChien\Lap 21`.
ZIP là **checkpoint**, chưa tự động là bài nộp hoàn chỉnh. Giữ nguyên bản gốc để đối chiếu; trợ lý sẽ đọc kết quả, hoàn thiện report và kiểm tra rubric trước khi tạo ZIP nộp.

In [ ]:
files.download(str(export_checkpoint('core_results')))

## 8. B1 tùy chọn — Merge và hot-swap
Chỉ bật sau khi core hoàn thành. NB6 lưu full merged weights tại adapters/merged: checkpoint chỉ lấy các file adapter, không lấy full weights. Mục thưởng còn cần log chứng minh >=2 adapter đã hot-swap.

In [ ]:
RUN_BONUS_B1 = False
if RUN_BONUS_B1:
    run_stage('nb6')
    files.download(str(export_checkpoint('bonus_b1')))
else:
    print('Chưa chạy B1; core đã có thể phân tích và viết report.')

## 9. Verify cuối — chỉ sau khi report đã hoàn thiện
Có thể chạy bây giờ để xem các mục còn thiếu, nhưng không gọi checkpoint là ready chỉ vì exit 0. Đọc từng WARN và đối chiếu đủ ví dụ định tính, kết luận >=150 từ, số liệu, phản tư.

In [ ]:
RUN_FINAL_VERIFY = False
if RUN_FINAL_VERIFY:
    run_stage('verify', ['scripts/verify.py'])
else:
    print('Verify nộp bài chờ report hoàn chỉnh.')